# Netra AI — Cataract vs Normal: Clean-Data Pipeline + MVP Training (Colab)
# নেত্র AI — ছানি বনাম সুস্থ চোখ: ডেটা পরিষ্কার + MVP ট্রেনিং (Colab)

**English.** This notebook (1) audits your image folder, (2) finds and removes duplicate / near-duplicate images (the original train/test split leaks), (3) filters bad images, (4) re-splits the data honestly, (5) checks for "shortcuts" (image size, brightness) the model could cheat with, (6) trains a small MobileNetV2 classifier, (7) reports honest metrics with confidence intervals, (8) draws Grad-CAM heatmaps, (9) exports a TFLite model, and (10) lets you test on your own real phone photos.

**বাংলা।** এই নোটবুক (১) আপনার ছবির ফোল্ডার পরীক্ষা করে, (২) কপি ও প্রায়-কপি ছবি খুঁজে বাদ দেয় (আসল train/test ভাগে ফাঁস আছে), (৩) খারাপ ছবি ছেঁটে ফেলে, (৪) ডেটা সৎভাবে নতুন করে ভাগ করে, (৫) এমন "শর্টকাট" আছে কি না দেখে যা দিয়ে মডেল চালাকি করতে পারে (ছবির মাপ, উজ্জ্বলতা), (৬) ছোট MobileNetV2 মডেল ট্রেন করে, (৭) আস্থার সীমাসহ সৎ ফল দেখায়, (৮) Grad-CAM হিটম্যাপ আঁকে, (৯) TFLite মডেল বের করে, আর (১০) আপনার নিজের তোলা ফোনের ছবিতে পরীক্ষা করতে দেয়।

> **Important / জরুরি.** The dataset is web-collected, not real phone photos from patients. Use results **only to test the pipeline**. Do **not** quote its accuracy in a paper or scholarship essay. The real test is your own phone photos with a doctor's labels (last section).
> ডেটাসেট ওয়েব থেকে সংগ্রহ করা, রোগীর ফোনের ছবি নয়। ফল দিয়ে **শুধু পাইপলাইন পরীক্ষা** করুন। এর নির্ভুলতা পেপারে বা স্কলারশিপ essay-তে **লিখবেন না**। আসল পরীক্ষা হলো ডাক্তারের লেবেলসহ আপনার নিজের ফোনের ছবি (শেষ অংশ)।

**How to run / কীভাবে চালাবেন:** Runtime → Change runtime type → **T4 GPU**. Then run cells top to bottom (Shift+Enter). / Runtime → Change runtime type → **T4 GPU** বেছে নিন, তারপর ওপর থেকে নিচে একটা একটা করে সেল চালান।

## 0. Settings / সেটিংস

**English.** Easiest route (no Google Drive needed): run `prepare_netra_upload.py` on your PC to make a small `Netra_AI_small.zip` (~50 MB), upload that zip to Colab's **Files** panel (folder icon on the left), and keep `ZIP_PATH` below. If you prefer Google Drive instead, set `USE_DRIVE = True` and point `DATA_ROOT` at your folder.

**বাংলা।** সবচেয়ে সহজ পথ (Google Drive লাগে না): আপনার পিসিতে `prepare_netra_upload.py` চালিয়ে ছোট `Netra_AI_small.zip` (~৫০ MB) বানান, সেটা Colab-এর বাঁ দিকের **Files** প্যানেলে (ফোল্ডার আইকন) আপলোড করুন, আর নিচের `ZIP_PATH` যেমন আছে তেমন রাখুন। Drive ব্যবহার করতে চাইলে `USE_DRIVE = True` করে `DATA_ROOT`-এ নিজের ফোল্ডারের পথ দিন।

In [ ]:
# ---- Where the data comes from / ডেটা কোথা থেকে আসবে ----
USE_DRIVE  = False                                  # True = use Google Drive paths below
ZIP_PATH   = "/content/Netra_AI_small.zip"          # the small zip you upload to Colab Files (set None to skip)
EXTRACT_TO = "/content"                             # the zip unpacks to /content/Netra_AI
DATA_ROOT  = "/content/Netra_AI"                    # folder containing train/ and test/
RAR_PATH   = None                                   # optional: original .rar (only if USE_DRIVE)
REAL_DIR   = "/content/Netra_real_phone"            # your own phone photos: cataract/ and normal/ (optional, last section)
OUT_DIR    = "/content/netra_outputs"               # results folder (download it at the end!)
WORK       = "/content/netra_work"                  # scratch folder on Colab disk

if USE_DRIVE:                                       # Drive versions of the paths / Drive-এর পথ
    DATA_ROOT = "/content/drive/MyDrive/Netra_AI"
    REAL_DIR  = "/content/drive/MyDrive/Netra_real_phone"
    OUT_DIR   = "/content/drive/MyDrive/Netra_outputs"

# ---- Classes / ক্লাস ----
POSITIVE, NEGATIVE = "cataract", "normal"          # positive = the disease we want to catch

# ---- Cleaning rules / পরিষ্কারের নিয়ম ----
MIN_SIDE    = 200    # drop images whose shorter side < 200 px (too tiny to see the lens)
MAX_ASPECT  = 2.7    # drop very wide images (side-by-side composites are ~3.0; single-eye crops are <2.5); review the grid below
HASH_DIST   = 4      # dHash distance <= 4 (of 64 bits) counts as "same picture"
MANUAL_EXCLUDE = []  # add relative paths you want removed, e.g. "train/cataract/image_147.jpg"

# ---- Training / ট্রেনিং ----
SEED = 42
IMG_SIZE = 224
MAX_SIDE_SAVE = 640  # cleaned copies are saved as JPEG with the longer side <= 640 (fast + small)
BATCH = 16
EPOCHS_HEAD = 15     # stage 1: train only the new top layer
EPOCHS_FT   = 15     # stage 2: fine-tune the last layers gently
UNFREEZE_LAST = 40   # how many of the last layers to fine-tune
WEIGHTS = "imagenet" # pretrained weights; None = random start (only for quick code tests)
MIN_SENSITIVITY = 0.95  # screening rule: catch at least 95% of cataract on validation data

In [ ]:
import os, glob, math, json, random, shutil, hashlib, subprocess, warnings, zipfile
import numpy as np, pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None
random.seed(SEED); np.random.seed(SEED)

# Google Drive only if requested / শুধু চাইলে Drive যুক্ত হবে
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        print("Not running in Colab - skipping Drive mount.")
    if RAR_PATH:                                    # optional: unpack the original .rar / ঐচ্ছিক: .rar খোলা
        subprocess.run("apt-get -qq install -y unrar", shell=True)
        subprocess.run(f'unrar x -y "{RAR_PATH}" "{os.path.dirname(DATA_ROOT)}/"', shell=True)

# Unpack the small zip you uploaded / আপলোড করা ছোট zip খোলা
if ZIP_PATH and os.path.isfile(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_TO)
    print("Unpacked", ZIP_PATH, "->", EXTRACT_TO)
elif ZIP_PATH and not USE_DRIVE:
    print("!! ZIP file not found at", ZIP_PATH, "- upload Netra_AI_small.zip to the Colab Files panel first.")
    print("!! ZIP পাওয়া যায়নি - আগে Netra_AI_small.zip Colab-এর Files প্যানেলে আপলোড করুন।")

print("DATA_ROOT exists:", os.path.isdir(DATA_ROOT))
os.makedirs(WORK, exist_ok=True)

## 1. Inventory — what is in the folder? / তালিকা: ফোল্ডারে কী আছে?

**English.** We list every image with its split, class, pixel size and file size. Huge differences in size between classes are an early warning sign of a shortcut.

**বাংলা।** প্রতিটি ছবির split, ক্লাস, পিক্সেল মাপ ও ফাইল সাইজ তালিকা করি। দুই ক্লাসের মাপে বড় ফারাক থাকলে সেটা "শর্টকাট"-এর প্রাথমিক সতর্কসংকেত।

In [ ]:
EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
rows = []
for split in ["train", "test"]:
    for cls in [NEGATIVE, POSITIVE]:
        for p in sorted(glob.glob(f"{DATA_ROOT}/{split}/{cls}/*")):
            if os.path.splitext(p)[1].lower() not in EXTS:
                continue
            try:
                with Image.open(p) as im:
                    w, h = im.size
            except Exception as e:
                print("Unreadable, skipped:", p, e); continue
            rows.append(dict(path=p, rel=os.path.relpath(p, DATA_ROOT).replace("\\", "/"),
                             orig_split=split, label=cls, w=w, h=h, bytes=os.path.getsize(p)))
df = pd.DataFrame(rows)
df["min_side"] = df[["w", "h"]].min(axis=1)
df["aspect"] = df[["w", "h"]].max(axis=1) / df["min_side"]
print("Total images:", len(df))
print(df.groupby(["orig_split", "label"]).size().unstack(fill_value=0))
print()
print(df.groupby("label")[["w", "h", "bytes"]].describe().T.round(0).to_string())

## 2. Find duplicates (exact + near) / কপি খোঁজা (হুবহু + প্রায়)

**English.** Two fingerprints per image: (a) SHA-256 of the file bytes = exact copies; (b) a 64-bit *difference hash* (dHash) of a shrunken grey picture = same picture re-saved, resized or renamed. Images linked by either fingerprint form a **group**; we keep only the best one per group. Groups whose members carry **different labels** are dropped completely (we can't trust them). We compare **all** images together — train and test — because the original split leaks.

**বাংলা।** প্রতিটি ছবির দুই ধরনের "আঙুলের ছাপ": (ক) ফাইলের SHA-256 = হুবহু কপি; (খ) ছোট করা সাদা-কালো ছবির ৬৪-বিটের dHash = একই ছবি নতুন নামে/মাপে সেভ করা। যেকোনো একটা ছাপ মিললে ছবিগুলো এক **গ্রুপ**, আর প্রতি গ্রুপ থেকে শুধু সেরাটা রাখি। যে গ্রুপের ছবিতে **ভিন্ন লেবেল** আছে, পুরো গ্রুপ বাদ। train ও test **একসাথে** মেলাই, কারণ আসল ভাগে ফাঁস আছে।

In [ ]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def dhash64(path):
    """64-bit difference hash: compares neighbouring pixels of a 9x8 grey thumbnail."""
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("L").resize((9, 8), Image.LANCZOS, reducing_gap=2.0)
    a = np.asarray(im, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    v = 0
    for b in bits:
        v = (v << 1) | int(b)
    return v

sha, dh = [], []
for i, p in enumerate(df["path"]):
    sha.append(sha256(p)); dh.append(dhash64(p))
    if (i + 1) % 100 == 0: print(f"  fingerprinted {i+1}/{len(df)}")
df["sha"] = sha
dh = np.array(dh, dtype=np.uint64)

# Union-find: link images that share a fingerprint / যে ছবিগুলোর ছাপ মেলে তাদের জোড়া লাগানো
n = len(df)
parent = list(range(n))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra

for _, idxs in df.groupby("sha").indices.items():          # exact copies
    for j in idxs[1:]: union(idxs[0], j)
for i in range(n - 1):                                     # near copies
    x = dh[i] ^ dh[i + 1:]
    dist = np.unpackbits(x.view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)
    for j in np.where(dist <= HASH_DIST)[0]:
        union(i, i + 1 + int(j))

df["group"] = [find(i) for i in range(n)]
g = df.groupby("group")
df["group_size"] = g["path"].transform("size")
df["group_labels"] = g["label"].transform("nunique")
df["group_splits"] = g["orig_split"].transform("nunique")

n_dup_groups = (g.size() > 1).sum()
extra = int((g.size() - 1).sum())
print(f"\nDuplicate groups: {n_dup_groups}  |  extra copies: {extra}  |  unique pictures: {df['group'].nunique()}")
print("Groups that appear in BOTH original train and test (leakage):", int(g["orig_split"].nunique().gt(1).sum()))
print("Groups with conflicting labels (dropped entirely):", int(g["label"].nunique().gt(1).sum()))

In [ ]:
def show_grid(paths, titles=None, cols=4, size=3.0):
    rows = math.ceil(len(paths) / cols)
    fig, axs = plt.subplots(rows, cols, figsize=(cols * size, rows * size))
    axs = np.array(axs).reshape(-1)
    for ax in axs: ax.axis("off")
    for ax, p, t in zip(axs, paths, titles or [""] * len(paths)):
        with Image.open(p) as im:
            im = ImageOps.exif_transpose(im).convert("RGB"); im.thumbnail((400, 400)); ax.imshow(im)
        ax.set_title(t, fontsize=8)
    plt.tight_layout(); plt.show()

# Look at NEAR (not byte-identical) groups to make sure the hash is not over-merging.
# শুধু "প্রায়-কপি" গ্রুপ দেখি, যাতে নিশ্চিত হই ভিন্ন ছবি ভুল করে এক হয়নি।
near = [gid for gid, sub in df.groupby("group") if sub["sha"].nunique() > 1]
print("Near-duplicate groups (different bytes, same picture):", len(near))
for gid in near[:4]:
    sub = df[df.group == gid].head(2)
    show_grid(sub["path"].tolist(), sub["rel"].tolist(), cols=2, size=3.5)

In [ ]:
# Keep the best image of each group (largest pixels, then largest file). Drop conflicting-label groups.
# প্রতি গ্রুপে সেরা ছবি রাখি (বেশি পিক্সেল)। ভিন্ন লেবেলের গ্রুপ পুরো বাদ।
df["pixels"] = df["w"] * df["h"]
df = df.sort_values(["group", "pixels", "bytes"], ascending=[True, False, False])
df["is_best"] = ~df.duplicated("group", keep="first")
df["drop_reason"] = ""
df.loc[~df["is_best"], "drop_reason"] = "duplicate"
df.loc[df["group_labels"] > 1, "drop_reason"] = "conflicting label"
df.to_csv(f"{WORK}/duplicates_report.csv", index=False)
print(df["drop_reason"].replace("", "kept so far").value_counts())

## 3. Quality filter / মান বাছাই

**English.** Remove images that are too small, extremely wide (likely side-by-side illustrations), or on your manual exclude list. Look at the grid of flagged images and adjust `MIN_SIDE`, `MAX_ASPECT` or `MANUAL_EXCLUDE` if needed. Also **look at random images from each class**: code cannot see watermarks, make-up, or age differences — your eyes must.

**বাংলা।** খুব ছোট, খুব চওড়া (সম্ভবত পাশাপাশি দুই ছবির চিত্র), বা আপনার হাতে দেওয়া বাদের তালিকার ছবি সরাই। চিহ্নিত ছবির গ্রিড দেখে দরকার হলে `MIN_SIDE`, `MAX_ASPECT`, `MANUAL_EXCLUDE` বদলান। প্রতিটি ক্লাস থেকে এলোমেলো ছবিও **নিজে দেখুন**: ওয়াটারমার্ক, মেকআপ বা বয়সের ফারাক কোড ধরতে পারে না, আপনার চোখ পারবে।

In [ ]:
cand = df[df["drop_reason"] == ""].copy()
too_small = cand["min_side"] < MIN_SIDE
too_wide  = cand["aspect"] > MAX_ASPECT
manual    = cand["rel"].isin(MANUAL_EXCLUDE)
df.loc[cand.index[too_small], "drop_reason"] = f"smaller than {MIN_SIDE}px"
df.loc[cand.index[too_wide & ~too_small], "drop_reason"] = f"aspect > {MAX_ASPECT}"
df.loc[cand.index[manual], "drop_reason"] = "manual exclude"

print(df["drop_reason"].replace("", "KEPT").value_counts())
flag = df[df["drop_reason"].str.startswith("aspect")]
if len(flag):
    print("\nWide images that were removed - check they really are composites / চওড়া ছবি যা বাদ গেছে:")
    show_grid(flag["path"].tolist()[:8], flag["rel"].tolist()[:8])

clean = df[df["drop_reason"] == ""].copy().reset_index(drop=True)
print("\nClean pool:", len(clean)); print(clean["label"].value_counts())

# Random look at each class / প্রতি ক্লাসের এলোমেলো ছবি
for cls in [POSITIVE, NEGATIVE]:
    s = clean[clean.label == cls].sample(min(8, (clean.label == cls).sum()), random_state=SEED)
    print(f"\nRandom '{cls}' images - look for watermarks, make-up, age differences:")
    show_grid(s["path"].tolist(), s["rel"].tolist())

## 4. Honest re-split / সৎভাবে নতুন ভাগ

**English.** Pool everything that survived and split **70 / 15 / 15** (train / validation / test), stratified by class. The test set is touched **once**, at the end. Note: web images have no patient ID, so we cannot guarantee that two different pictures of the same person are kept apart — another reason this is only a pipeline test.

**বাংলা।** যা টিকে গেছে সব একসাথে করে **৭০ / ১৫ / ১৫** (train / validation / test) ভাগ করি, ক্লাস অনুপাত ঠিক রেখে। test সেট শেষে **একবারই** ব্যবহার হবে। মনে রাখবেন, ওয়েবের ছবিতে রোগীর আইডি নেই, তাই একই মানুষের ভিন্ন দুই ছবি আলাদা ভাগে পড়েছে কি না নিশ্চিত করা যায় না। এটাও কারণ কেন এটা শুধু পাইপলাইন পরীক্ষা।

In [ ]:
from sklearn.model_selection import train_test_split
idx_tr, idx_tmp = train_test_split(clean.index, test_size=0.30, stratify=clean["label"], random_state=SEED)
idx_va, idx_te = train_test_split(idx_tmp, test_size=0.50, stratify=clean.loc[idx_tmp, "label"], random_state=SEED)
clean["split"] = ""
clean.loc[idx_tr, "split"] = "train"; clean.loc[idx_va, "split"] = "val"; clean.loc[idx_te, "split"] = "test"
print(clean.groupby(["split", "label"]).size().unstack(fill_value=0))

CLEAN = f"{WORK}/clean"
shutil.rmtree(CLEAN, ignore_errors=True)
for i, r in clean.iterrows():
    out_dir = f"{CLEAN}/{r.split}/{r.label}"; os.makedirs(out_dir, exist_ok=True)
    with Image.open(r.path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        im.thumbnail((MAX_SIDE_SAVE, MAX_SIDE_SAVE))
        im.save(f"{out_dir}/{r.label}_{i:04d}.jpg", quality=95)
    clean.loc[i, "clean_path"] = f"{out_dir}/{r.label}_{i:04d}.jpg"
clean.to_csv(f"{WORK}/clean_manifest.csv", index=False)
print("Saved cleaned copies to", CLEAN)

## 5. Shortcut check / শর্টকাট পরীক্ষা

**English.** We train a tiny model that sees **no eye at all** — only the image's width, height, file size and average colour. If it can still tell the classes apart (AUC well above 0.5), then the dataset has shortcuts and a neural network can cheat the same way. AUC ≈ 0.5 is what we want.

**বাংলা।** আমরা এমন একটা ছোট মডেল ট্রেন করি যে **চোখ একদমই দেখে না**, শুধু ছবির প্রস্থ, উচ্চতা, ফাইল সাইজ আর গড় রঙ জানে। এতেও যদি দুই ক্লাস আলাদা করতে পারে (AUC ০.৫-এর অনেক ওপরে), তার মানে ডেটায় শর্টকাট আছে এবং নিউরাল নেটওয়ার্কও একইভাবে চালাকি করতে পারবে। আমরা চাই AUC ≈ ০.৫।

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score

def colour_stats(p):
    with Image.open(p) as im:
        a = np.asarray(im.convert("RGB").resize((64, 64)), dtype=np.float32) / 255.0
    return [a[..., 0].mean(), a[..., 1].mean(), a[..., 2].mean(), a.mean(), a.std()]

X_meta = np.c_[clean["w"], clean["h"], clean["aspect"], np.log(clean["bytes"]), clean["bytes"] / clean["pixels"]]
X_col  = np.array([colour_stats(p) for p in clean["clean_path"]])
y_all  = (clean["label"] == POSITIVE).astype(int).values
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
for name, X in [("image size / file size only", X_meta), ("average colour only", X_col), ("both", np.c_[X_meta, X_col])]:
    pr = cross_val_predict(GradientBoostingClassifier(random_state=SEED), X, y_all, cv=skf, method="predict_proba")[:, 1]
    print(f"{name:32s} AUC = {roc_auc_score(y_all, pr):.3f}")
print("\nAUC near 0.5 = no obvious shortcut. AUC above ~0.70 = the data itself gives the answer away; treat any model score with suspicion.")
print("AUC ০.৫-এর কাছে = স্পষ্ট শর্টকাট নেই। ~০.৭০-এর ওপরে = ডেটা নিজেই উত্তর বলে দিচ্ছে; মডেলের স্কোরে সন্দেহ রাখুন।")

## 6. Data loaders + model / ডেটা লোডার ও মডেল

**English.** MobileNetV2 (small enough for a phone) with ImageNet weights. Augmentation (flip, small rotation/zoom, brightness/contrast) is part of the model and switches itself off at prediction time. The pixel scaling is also inside the model, so the phone app must feed **raw 0-255 pixels** (no manual normalisation). Output = probability of cataract.

**বাংলা।** ImageNet ওয়েটসহ MobileNetV2 (ফোনের জন্য যথেষ্ট ছোট)। augmentation (উল্টানো, সামান্য ঘোরানো/জুম, উজ্জ্বলতা/কনট্রাস্ট) মডেলের ভেতরেই আছে, আর অনুমান করার সময় নিজে থেকে বন্ধ থাকে। পিক্সেল স্কেলিংও মডেলের ভেতরে, তাই ফোন অ্যাপকে **কাঁচা ০–২৫৫ পিক্সেল** দিতে হবে (আলাদা normalisation নয়)। আউটপুট = ছানির সম্ভাবনা।

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

def make_ds(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        f"{CLEAN}/{split}", labels="inferred", label_mode="binary",
        class_names=[NEGATIVE, POSITIVE],          # normal = 0, cataract = 1
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, shuffle=shuffle, seed=SEED
    ).prefetch(tf.data.AUTOTUNE)

ds_tr, ds_va, ds_te = make_ds("train", True), make_ds("val", False), make_ds("test", False)

n_pos = int((clean[clean.split == "train"].label == POSITIVE).sum()); n_neg = int((clean[clean.split == "train"].label == NEGATIVE).sum())
class_weight = {0: (n_pos + n_neg) / (2 * n_neg), 1: (n_pos + n_neg) / (2 * n_pos)}
print("class weights:", class_weight)

def build_model():
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = layers.RandomFlip("horizontal")(inp)
    x = layers.RandomRotation(0.06)(x)
    x = layers.RandomZoom(0.10)(x)
    x = layers.RandomContrast(0.15)(x)
    x = layers.RandomBrightness(0.15, value_range=(0, 255))(x)
    x = layers.Rescaling(1 / 127.5, offset=-1, name="rescale")(x)
    base = tf.keras.applications.MobileNetV2(input_tensor=x, input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=WEIGHTS)
    x = layers.GlobalAveragePooling2D(name="gap")(base.output)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid", name="p_cataract")(x)
    return models.Model(inp, out)

def set_trainable(model, n_unfreeze):
    names = [l.name for l in model.layers]
    gap_i, resc_i = names.index("gap"), names.index("rescale")
    for i, l in enumerate(model.layers):
        if i <= resc_i: continue
        if i < gap_i:   # MobileNetV2 body: BatchNorm always frozen / BatchNorm সবসময় ফ্রিজ
            l.trainable = (i >= gap_i - n_unfreeze) and not isinstance(l, layers.BatchNormalization)
        else:
            l.trainable = True

def compile_model(model, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

model = build_model()
print("Parameters:", f"{model.count_params():,}")

## 7. Train in two stages / দুই ধাপে ট্রেনিং

**English.** Stage 1 trains only the new top layer (fast, stable). Stage 2 gently fine-tunes the last layers with a much smaller learning rate. Early stopping watches **validation** AUC and restores the best weights. The test set is not used here.

**বাংলা।** ধাপ ১-এ শুধু নতুন ওপরের স্তর ট্রেন হয় (দ্রুত, স্থির)। ধাপ ২-এ অনেক ছোট learning rate দিয়ে শেষের কিছু স্তর আস্তে আস্তে ঠিক করা হয়। Early stopping **validation** AUC দেখে, আর সেরা ওয়েট ফিরিয়ে আনে। এখানে test সেট ছোঁয়া হয় না।

In [ ]:
def callbacks():
    return [tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)]

# Stage 1: head only / শুধু ওপরের স্তর
set_trainable(model, 0); compile_model(model, 1e-3)
h1 = model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS_HEAD, class_weight=class_weight, callbacks=callbacks(), verbose=2)

# Stage 2: fine-tune last layers / শেষের স্তরগুলো ফাইন-টিউন
set_trainable(model, UNFREEZE_LAST); compile_model(model, 1e-5)
h2 = model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS_FT, class_weight=class_weight, callbacks=callbacks(), verbose=2)

hist = {k: h1.history[k] + h2.history[k] for k in h1.history}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist["loss"], label="train"); ax[0].plot(hist["val_loss"], label="val"); ax[0].set_title("loss"); ax[0].legend()
ax[1].plot(hist["auc"], label="train"); ax[1].plot(hist["val_auc"], label="val"); ax[1].set_title("AUC"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Honest evaluation / সৎ মূল্যায়ন

**English.** For a *screening* tool, missing a cataract is worse than a false alarm, so we choose the decision threshold **on the validation set** so that sensitivity is at least `MIN_SENSITIVITY`, then report the test result once. Every rate comes with a 95% confidence interval (Wilson). With so few test images the intervals are wide — that is the honest picture.

**বাংলা।** *স্ক্রিনিং* টুলে ছানি ধরতে না পারা ভুল সতর্কতার চেয়ে খারাপ, তাই সিদ্ধান্তের সীমা (threshold) **validation সেটে** এমনভাবে ঠিক করি যাতে sensitivity কমপক্ষে `MIN_SENSITIVITY` থাকে, তারপর test-এর ফল একবার দেখাই। প্রতিটি হারের সাথে ৯৫% আস্থার সীমা (Wilson) আছে। test ছবি কম বলে সীমা চওড়া হবে, এটাই সৎ চিত্র।

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve

def get_probs(m, ds):
    p = m.predict(ds, verbose=0).ravel()
    y = np.concatenate([b[1].numpy().ravel() for b in ds]).astype(int)
    return y, p

def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n)
    a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - a) / d, (c + a) / d

def choose_threshold(y, p, min_sens):
    fpr, tpr, thr = roc_curve(y, p)
    ok = np.where(tpr >= min_sens)[0]
    if len(ok) == 0: return 0.5
    best = ok[np.argmin(fpr[ok])]
    t = float(thr[best]); return min(max(t, 0.0), 1.0)

def report(y, p, thr, title):
    tn, fp, fn, tp = confusion_matrix(y, (p >= thr).astype(int), labels=[0, 1]).ravel()
    rows = {"Sensitivity (cataract caught)": (tp, tp + fn), "Specificity (normal cleared)": (tn, tn + fp),
            "PPV (alarm is right)": (tp, tp + fp), "NPV (all-clear is right)": (tn, tn + fn),
            "Accuracy": (tp + tn, len(y))}
    print(f"\n=== {title} | threshold = {thr:.3f} | n = {len(y)} (cataract {tp+fn}, normal {tn+fp}) ===")
    print(f"TP={tp}  FN={fn}  FP={fp}  TN={tn}")
    for k, (a, b) in rows.items():
        lo, hi = wilson(a, b)
        print(f"{k:32s} {a/b if b else float('nan'):.3f}   95% CI [{lo:.3f}, {hi:.3f}]   ({a}/{b})")
    try: print(f"{'ROC AUC':32s} {roc_auc_score(y, p):.3f}")
    except ValueError: pass

y_va, p_va = get_probs(model, ds_va)
THRESH = choose_threshold(y_va, p_va, MIN_SENSITIVITY)
report(y_va, p_va, THRESH, "VALIDATION (used to choose the threshold)")
y_te, p_te = get_probs(model, ds_te)
report(y_te, p_te, THRESH, "TEST (look only once)")

fpr, tpr, _ = roc_curve(y_te, p_te)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "--", c="grey"); ax[0].set_xlabel("1 - specificity"); ax[0].set_ylabel("sensitivity"); ax[0].set_title("ROC (test)")
cm = confusion_matrix(y_te, (p_te >= THRESH).astype(int), labels=[0, 1])
ax[1].imshow(cm, cmap="Blues"); ax[1].set_xticks([0, 1]); ax[1].set_yticks([0, 1])
ax[1].set_xticklabels(["pred normal", "pred cataract"]); ax[1].set_yticklabels(["true normal", "true cataract"])
for (i, j), v in np.ndenumerate(cm): ax[1].text(j, i, int(v), ha="center", va="center", fontsize=14)
ax[1].set_title("Confusion matrix (test)"); plt.tight_layout(); plt.show()
print("\nREMINDER: this is a pipeline test on web images. Do not quote these numbers as field performance.")
print("মনে রাখুন: এটা ওয়েব ছবির ওপর পাইপলাইন পরীক্ষা। এই সংখ্যাকে মাঠের কার্যকারিতা বলে উল্লেখ করবেন না।")

## 9. Grad-CAM — where is the model looking? / মডেল কোথায় দেখছে?

**English.** Grad-CAM highlights the image regions that pushed the score towards "cataract". For cataract images the hot area should sit on the **pupil / lens**. If it lights up the eyelids, skin or background, the model is using a shortcut. A heatmap is a sanity check, not proof of diagnosis.

**বাংলা।** Grad-CAM দেখায় ছবির কোন অংশ স্কোরকে "ছানি"-র দিকে ঠেলেছে। ছানির ছবিতে উজ্জ্বল জায়গা হওয়া উচিত **মণি / লেন্সে**। যদি চোখের পাতা, ত্বক বা পটভূমি জ্বলজ্বল করে, তাহলে মডেল শর্টকাট ব্যবহার করছে। হিটম্যাপ একটা যুক্তিসঙ্গত যাচাই মাত্র, রোগ নির্ণয়ের প্রমাণ নয়।

In [ ]:
grad_model = tf.keras.Model(model.inputs, [model.get_layer("out_relu").output, model.output])

def grad_cam(batch):
    batch = tf.convert_to_tensor(batch, tf.float32)
    with tf.GradientTape() as tape:
        conv, pred = grad_model(batch, training=False)
        score = pred[:, 0]
    grads = tape.gradient(score, conv)
    w = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv * w[:, None, None, :], axis=-1))
    cam = cam / (tf.reduce_max(cam, axis=(1, 2), keepdims=True) + 1e-8)
    return cam.numpy(), pred.numpy().ravel()

def show_cam(ds, n=8):
    imgs, ys = next(iter(ds.unbatch().shuffle(200, seed=SEED).batch(n)))
    cams, preds = grad_cam(imgs.numpy())
    fig, axs = plt.subplots(2, n, figsize=(2.4 * n, 5))
    for i in range(n):
        im = imgs[i].numpy().astype("uint8")
        cam = np.array(Image.fromarray((cams[i] * 255).astype("uint8")).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)) / 255.0
        axs[0, i].imshow(im); axs[0, i].axis("off")
        axs[0, i].set_title(f"true={'cataract' if ys[i] > .5 else 'normal'}\np={preds[i]:.2f}", fontsize=8)
        axs[1, i].imshow(im); axs[1, i].imshow(cam, cmap="jet", alpha=0.45); axs[1, i].axis("off")
    plt.tight_layout(); plt.show()

show_cam(ds_te, n=min(8, BATCH))

## 10. Export to TFLite (for the phone app) / TFLite এক্সপোর্ট (ফোন অ্যাপের জন্য)

**English.** Converts the model to a small `.tflite` file (dynamic-range quantisation) and checks that it gives the same answers as the Keras model. **Input for the app:** a 224×224 RGB image as **float32 with raw pixel values 0–255**. **Output:** one number = probability of cataract; compare it with the saved threshold.

**বাংলা।** মডেলকে ছোট `.tflite` ফাইলে রূপান্তর করে (dynamic-range quantisation) এবং Keras মডেলের সাথে উত্তর মেলে কি না যাচাই করি। **অ্যাপের ইনপুট:** ২২৪×২২৪ RGB ছবি, **float32, কাঁচা ০–২৫৫ পিক্সেল মান**। **আউটপুট:** একটা সংখ্যা = ছানির সম্ভাবনা; সেটা সংরক্ষিত threshold-এর সাথে মেলান।

In [ ]:
import logging
logging.getLogger("absl").setLevel(logging.ERROR); tf.get_logger().setLevel("ERROR")   # hide converter chatter / কনভার্টারের বাড়তি লেখা লুকানো
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    tfl = conv.convert()
tfl_path = f"{WORK}/netra_cataract.tflite"
open(tfl_path, "wb").write(tfl)
print(f"TFLite size: {len(tfl)/1e6:.2f} MB")

interp = tf.lite.Interpreter(model_content=tfl); interp.allocate_tensors()
inp_d, out_d = interp.get_input_details()[0], interp.get_output_details()[0]
imgs, _ = next(iter(ds_te))
diffs = []
for im in imgs.numpy()[:8]:
    interp.set_tensor(inp_d["index"], im[None].astype(np.float32)); interp.invoke()
    p_lite = float(interp.get_tensor(out_d["index"]).ravel()[0])
    p_keras = float(model.predict(im[None], verbose=0).ravel()[0])
    diffs.append(abs(p_lite - p_keras))
print("Max |TFLite - Keras| probability difference:", round(max(diffs), 4), "(should be small, e.g. < 0.02)")

## 11. Save everything / সব সংরক্ষণ

**বাংলা।** মডেল, TFLite ফাইল, threshold আর পরিষ্কারের রিপোর্ট আপনার Drive-এ কপি হবে, যাতে Colab বন্ধ হলেও থাকে। / **English.** Model, TFLite, threshold and cleaning report are copied to your Drive so they survive a Colab reset.

In [ ]:
model.save(f"{WORK}/netra_mobilenetv2.keras")
meta = {"threshold": THRESH, "min_sensitivity_target": MIN_SENSITIVITY, "img_size": IMG_SIZE,
        "input": "float32 RGB 0-255 (no normalisation)", "positive_class": POSITIVE,
        "n_clean": int(len(clean)), "seed": SEED,
        "warning": "Trained on web images; pipeline test only. Not validated on real phone photos."}
json.dump(meta, open(f"{WORK}/model_meta.json", "w"), indent=2)
try:
    os.makedirs(OUT_DIR, exist_ok=True)
    for f in ["netra_mobilenetv2.keras", "netra_cataract.tflite", "model_meta.json", "duplicates_report.csv", "clean_manifest.csv"]:
        shutil.copy(f"{WORK}/{f}", OUT_DIR)
    print("Copied to", OUT_DIR)
except Exception as e:
    print("Could not copy to Drive:", e)

### Download your results / ফলাফল ডাউনলোড

**বাংলা।** Colab বন্ধ হলে ফাইল মুছে যায়, তাই এখনই zip করে ডাউনলোড করুন। / **English.** Colab wipes files when the session ends, so zip and download them now.

In [ ]:
if not USE_DRIVE:
    shutil.make_archive("/content/netra_results", "zip", OUT_DIR)
    try:
        from google.colab import files
        files.download("/content/netra_results.zip")
    except ImportError:
        print("Results zip:", "/content/netra_results.zip")

## 12. The REAL test — your own phone photos / আসল পরীক্ষা — আপনার নিজের ফোনের ছবি

**English.** Put your pilot photos in `REAL_DIR/cataract/` and `REAL_DIR/normal/`, labelled by the **ophthalmologist's** diagnosis (with patient consent and the required approval). This set was never seen in training or in choosing the threshold, so its result is the only one worth reporting — with its wide confidence interval. If it is much worse than section 8, that gap is the finding: the web data does not match real field photos.

**বাংলা।** পাইলটের ছবিগুলো `REAL_DIR/cataract/` ও `REAL_DIR/normal/`-এ রাখুন, **চক্ষু চিকিৎসকের** রায় অনুযায়ী লেবেল দিয়ে (রোগীর সম্মতি ও প্রয়োজনীয় অনুমোদনসহ)। এই সেট ট্রেনিংয়ে বা threshold ঠিক করায় কখনো ব্যবহার হয়নি, তাই শুধু এর ফলই রিপোর্ট করার যোগ্য, চওড়া আস্থার সীমাসহ। সেকশন ৮-এর চেয়ে ফল অনেক খারাপ হলে সেই ফারাকই আপনার আবিষ্কার: ওয়েবের ডেটা মাঠের ছবির সাথে মেলে না।

In [ ]:
if os.path.isdir(f"{REAL_DIR}/{POSITIVE}") and os.path.isdir(f"{REAL_DIR}/{NEGATIVE}"):
    ds_real = tf.keras.utils.image_dataset_from_directory(
        REAL_DIR, labels="inferred", label_mode="binary", class_names=[NEGATIVE, POSITIVE],
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, shuffle=False)
    y_r, p_r = get_probs(model, ds_real)
    report(y_r, p_r, THRESH, "REAL PHONE PHOTOS (external test)")
    show_cam(ds_real, n=min(8, BATCH))
else:
    print(f"No real-photo folder yet at {REAL_DIR} - create {POSITIVE}/ and {NEGATIVE}/ inside it when you have doctor-labelled photos.")
    print("এখনো আসল ছবির ফোল্ডার নেই। ডাক্তারের লেবেলসহ ছবি পেলে সেখানে cataract/ ও normal/ ফোল্ডার বানান।")

## What you may and may not claim / কী লিখতে পারবেন, কী পারবেন না

| | English | বাংলা |
|---|---|---|
| ✅ May say | "We built a reproducible pipeline: de-duplication, honest re-split, shortcut audit, MobileNetV2, TFLite, Grad-CAM." | "আমরা একটা পুনরায় চালানো যায় এমন পাইপলাইন বানিয়েছি: কপি বাদ, সৎ ভাগ, শর্টকাট পরীক্ষা, MobileNetV2, TFLite, Grad-CAM।" |
| ✅ May say | "Original split leaked: N test images were copies of training images." | "আসল ভাগে ফাঁস ছিল: N টি test ছবি train-এর কপি।" |
| ❌ Do not say | "Our model reaches X% accuracy for cataract detection." (web images) | "আমাদের মডেল ছানি শনাক্তে X% নির্ভুল।" (ওয়েব ছবির ওপর) |
| ✅ Only after section 12 | Sensitivity / specificity **with 95% CI** on doctor-labelled phone photos, stating n. | ডাক্তারের লেবেলসহ ফোনের ছবিতে **৯৫% আস্থার সীমাসহ** sensitivity / specificity, n উল্লেখ করে। |

**Screening aid only** — it does not diagnose. A positive result means "see an eye-care professional". / **শুধু স্ক্রিনিং সহায়ক**, রোগ নির্ণয় নয়। পজিটিভ ফলের মানে "চক্ষু চিকিৎসককে দেখান"।